# **Project 2 - LLMs and Generative AI**

Below is a ready-to-use notebook outline for your assignment. Students will only need to change the prompts in designated cells. The notebook uses Hugging Face Transformers for open-source models, namely MedGemma.

## Install and Import Packages

In [40]:
# Package requirements for the notebook to run successfully
!pip install bitsandbytes

In [41]:
# Import necessary libraries for the project
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    pipeline
)
import torch

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Load Generation Model and Tokenizer


**MedGemma Description:**

MedGemma is a collection of Gemma 3 variants that are trained for performance on medical text and image comprehension.

You can find more information here:
https://huggingface.co/unsloth/medgemma-4b-it-bnb-4bit.

In [42]:
# Define the name of the model to be loaded from Hugging Face
model_name = "unsloth/medgemma-4b-it-bnb-4bit"

# Print status message to let the user know which model is being loaded
print(f"🔄 Loading generation model: {model_name}")

# Load the language model (LLM) for text generation
generation_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, # define the type of data to use if gpu is available
    device_map="auto"
)

# Load the tokenizer for the model
# The tokenizer is responsible for converting text to tokens the model can understand (and back to text)
generation_tokenizer = AutoTokenizer.from_pretrained(model_name)
generation_tokenizer.pad_token = generation_tokenizer.pad_token or generation_tokenizer.eos_token

🔄 Loading generation model: unsloth/medgemma-4b-it-bnb-4bit


Function to generate the output using the LLM defined above

In [43]:
def generate(messages):

    # Convert a list of chat messages into a single string using the model's chat template.
    # This prepares the conversation in the format the model expects (e.g., <user>, <assistant> tags).
    text = generation_tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

    # Tokenize the input text and prepare it for the model.
    inputs = generation_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=1000).to(generation_model.device)

    # Generate output using the model
    with torch.inference_mode():
        generation = generation_model.generate(
            **inputs, max_new_tokens=300, do_sample=False,
            pad_token_id=generation_tokenizer.eos_token_id
        )

    # Decode the model's output, transforming the token IDs back into a human-readable string.
    output = generation_tokenizer.decode(generation[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    return output

## Load Notes

**Note 1:** English clinical note

In [44]:
note = """The patient is a 72-year-old female with a past medical history of atrial fibrillation, chronic kidney disease stage 3, hypertension, and osteoarthritis.
She presented to the ED with complaints of generalized weakness, lightheadedness, and two episodes of syncope over the past 24 hours.

On exam, she was hypotensive (BP 88/56) and bradycardic with an irregularly irregular pulse. Labs showed elevated BUN and creatinine, consistent with worsening CKD, and mild hyperkalemia.
ECG revealed atrial fibrillation with slow ventricular response. Chest X-ray showed no acute pulmonary findings.
The patient was admitted to telemetry for further monitoring. Her beta-blocker (metoprolol) was held, and nephrology was consulted for worsening renal function. C
ardiology was consulted for bradyarrhythmia management. A temporary transvenous pacemaker was placed.

She was monitored closely, and after stabilization, underwent permanent pacemaker implantation on hospital day 3. Post-op course was uncomplicated.
Renal function improved with IV fluids and adjustment of antihypertensive medications.

Discharge medications included: apixaban, lisinopril (dose reduced), and acetaminophen for pain. She was advised to follow up with cardiology and nephrology within one week.
Discharged home with stable vital signs and improved energy levels."""

**Note 2:** Portuguese clinical note

In [45]:
note_pt = """1.a consulta
NHC 118441566
H, 67 anos, de Sesimbra, mestre pesca reformado (desde 58 anos)
Sempre trabalhou de noite e dormiu de dia. Dormia cerca de 3 h.
Desde que se reformou tem dificuldade em dormir. Dorme 3-4 h
Tem uma empresa. Por vezes ansioso
Faz alprazolan 0,25 mg ao jantar. Deita-se 21h-22h. Leva cerca de 2 horas para adormecer. Dorme 3-4 h. ACorda e não sabe porquê. Fica na cama e depois levanta-se.
Antes
Já fez mirtazapina 15 mg, durante 1 Mês e acha que não teve efeito terapeutico
DRC por provável pielonefrite crónica litiásica (hiperuricémia?), inicio de hemodiálise em julho 2002, Transplante renal de dador vivo a 17/09/2002. Reinicio de hemodiálise por disfunção crónica do enxerto a 17/08/2016
HTA secundária
Hábitos tabágicos suspensos 2002

Adenocrcinoma da prostata, submetido a prostatectomia total em janeiro de 2018

Tumor Septissémia por bactéria provocada por um peixe... em set 2018. Ferida no MS dto...

Medicação atual: amlodipina e concor . Alprazolam

Refere sensação de calor nos membros inferiores que melhora com movimento, piora no final do dia, por vezes dificultam o adormecer

Roncopatia. Sem apneias identificadas.

Xavier Mourato OM 52847"""

## **Exercise 1: Summarization**

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 1.1 - Intrepertation of the Clinical Note Summary
  <p>Consider the following example for summarize the clinical note.
  
  What can you say about the summary generated? Is the model hallucinating?
  </p>
</div>


In [46]:
# This variable will be our prompt for summarization.
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant."
    },
    {
        "role": "user",
        "content": f"""Summarize the following clinical note:\n\nCLINICAL NOTE:\n{note}"""
    }
]

# Here we will generate an output for the prompt using the generator function
output = generate(messages)
print(output)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


A 72-year-old female with a history of atrial fibrillation, chronic kidney disease (CKD), hypertension, and osteoarthritis presented to the ED with weakness, lightheadedness, and syncope. She was found to be hypotensive and bradycardic with atrial fibrillation. Labs showed elevated BUN and creatinine, and mild hyperkalemia. An ECG showed atrial fibrillation with slow ventricular response. A temporary pacemaker was placed, and a permanent pacemaker was implanted on hospital day 3. Her renal function improved with IV fluids and medication adjustments. She was discharged home with stable vital signs, improved energy levels, and follow-up appointments with cardiology and nephrology.




- O resumo cobre os pontos principais da nota clínica
- Não ocorreram halucinações por parte do modelo
- Pode haver omissões subtis, como detalhes dos exames ou histórico mais antigo.

O  modelo foi razoavelmente confiável nesta tarefa. No entanto, ainda exige validação humana antes de uso clínico.



<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 1.2 - Structured Clinical Notes
  <p>What if we want to structure the clinical Note?
  
  Change the prompt presented in the example before to have the summary of the clinical note in a structured format.
  
  Example of a structured clinical note:
  * Demographics:
  * History:
  * Diagnosis:
  * ...
  </p>
</div>

In [47]:

messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant."
    },
    {
        "role": "user",
        "content": f"""Summarize the following clinical note using the following structure:
* Demographics:
* Chief Complaint:
* History of Present Illness:
* Vital Signs:
* Physical Exam:
* Labs/Imaging:
* Diagnosis:
* Treatment Plan:
* Discharge Medications:
* Follow-up Instructions:

CLINICAL NOTE:
{note}"""
    }
]


output= generate(messages)
print(output)


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Here's a summary of the clinical note, following the requested structure:

*   **Demographics:** 72-year-old female
*   **Chief Complaint:** Generalized weakness, lightheadedness, two episodes of syncope
*   **History of Present Illness:** The patient presented to the ED with generalized weakness, lightheadedness, and two episodes of syncope over the past 24 hours.
*   **Vital Signs:** Hypotensive (BP 88/56), bradycardic with an irregularly irregular pulse.
*   **Physical Exam:**
    *   ECG: Atrial fibrillation with slow ventricular response.
    *   Chest X-ray: No acute pulmonary findings.
*   **Labs/Imaging:**
    *   Labs: Elevated BUN and creatinine (consistent with worsening CKD), mild hyperkalemia.
    *   ECG: Atrial fibrillation with slow ventricular response.
    *   Chest X-ray: No acute pulmonary findings.
*   **Diagnosis:** Atrial fibrillation with slow ventricular response, likely contributing to syncope and hypotension. Worsening chronic kidney disease (CKD).
*   **Trea

## **Exercise 2: Question-Answering**
We can use the LLM to answer specific questions regarding the clinical note.

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 2.1 - Patient status on Exam
  <p>How was the patient feeling in the moment of the exame?

  Change the prompt and interpret the output.
  </p>
</div>

In [48]:

messages = [
    {
        "role": "system",
        "content": "You are a helpful clinical assistant."
    },
    {
        "role": "user",
        "content": f"""Based on the following clinical note, how was the patient feeling at the time of the physical exam? Include any vital signs or symptoms observed.

CLINICAL NOTE:
{note}"""
    }
]


output = generate(messages)
print(output)


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Based on the clinical note, the patient was feeling **generalized weakness, lightheadedness, and was hypotensive (BP 88/56) and bradycardic with an irregularly irregular pulse** at the time of the physical exam.




Neste exercício, usei um prompt para perguntar ao modelo como o paciente estava no momento do exame físico.

A resposta gerada foi coerente. O modelo identificou corretamente que a paciente apresentava hipotensão (pressão arterial de 88/56 mmHg), bradicardia e pulso irregular. Além disso, mencionou sintomas como fraqueza e tontura, que realmente estão descritos na nota clínica.

O modelo não inventou informações e respondeu de forma clara, com base nos dados presentes no texto.

O modelo respondeu bem à pergunta e demonstrou uma boa capacidade de interpretação clínica básica. Mesmo assim, como em qualquer aplicação de IA na área da saúde, é fundamental que as respostas sejam sempre revistas por médicos



<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 2.2 - Discharge Medication
  <p>Which drugs are included in the discharge medications?

  Change the prompt and interpret the output. Try include the doses if they are available.
  </p>
</div>

In [49]:

messages = [
    {
        "role": "system",
        "content": "You are a helpful clinical assistant."
    },
    {
        "role": "user",
        "content": f"""Based on the following clinical note, which drugs were prescribed at discharge?
If the note includes doses, list them as well.

CLINICAL NOTE:
{note}"""
    }
]


output = generate(messages)
print(output)


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following medications were prescribed at discharge:

*   **Apixaban**
*   **Lisinopril** (dose reduced)
*   **Acetaminophen**



Neste exercício, perguntei ao modelo quais medicamentos foram prescritos na alta da paciente, com a indicação de que incluísse as doses, se disponíveis na nota.

O modelo respondeu corretamente que os medicamentos de alta foram:
- Apixaban
- Lisinopril (com a observação de que a dose foi reduzida)
- Acetaminophen

A resposta está alinhada com o que foi descrito na nota clínica. O modelo não inventou doses e respeitou a informação original, onde só é mencionado que o lisinopril teve a dose ajustada, sem especificar valores. Isso mostra que o modelo evitou fazer suposições, o que é positivo.

A resposta foi clara, direta e clinicamente adequada. O modelo conseguiu identificar corretamente os medicamentos prescritos na alta, demonstrando utilidade para tarefas simples de extração de informação. Como sempre, seria necessário confirmar manualmente em um cenário clínico real.


## **Exercise 3 - Entity Extraction**

<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 3.1 - Procedures Extraction
  <p>Still using the same LLM, find the procedures mentioned in the clinical note?

  Change the prompt and interpret the output.
  </p>
</div>

In [50]:

messages = [
    {
        "role": "system",
        "content": "You are a helpful clinical assistant."
    },
    {
        "role": "user",
        "content": f"""From the clinical note below, extract all the medical procedures the patient underwent.
List them clearly using bullet points.

CLINICAL NOTE:
{note}"""
    }
]


output = generate(messages)
print(output)


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Here are the medical procedures the patient underwent, extracted from the clinical note:

*   Temporary transvenous pacemaker placement
*   Permanent pacemaker implantation



<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 3.2 - Conditions Extraction for clinical note in portuguese
  <p>Still using the same LLM, find the conditions mentioned in the clinical note written in portuguese?

  Change the prompt and interpret the output. The portuguese note is saved in the variable 'note_pt'.
  </p>
</div>

In [51]:

messages = [
    {
        "role": "system",
        "content": "Você é um assistente clínico que entende e responde em português."
    },
    {
        "role": "user",
        "content": f"""A partir da nota clínica abaixo, identifique todas as condições médicas mencionadas.
Liste as condições com marcadores (bullet points).

NOTA CLÍNICA:
{note_pt}"""
    }
]


output = generate(messages)
print(output)


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Com base na nota clínica fornecida, as condições médicas mencionadas são:

*   **Insônia:** Dificuldade em dormir, com duração de 3-4 horas por noite.
*   **Ansiedade:** Refere-se a momentos de ansiedade.
*   **Pielonefrite crônica litiásica (hiperuricémia?):** Histórico de problemas renais, incluindo hiperuricemia.
*   **Disfunção crônica do enxerto (DCE):** Reinstato de hemodiálise devido à disfunção do transplante renal.
*   **Hipertensão arterial (HTA) secundária:** HTA causada por outra condição médica.
*   **Adenocarcinoma da próstata:** Diagnóstico anterior, com tratamento por prostatectomia total.
*   **Septissémia por bactéria:** Infecção bacteriana no sangue, relacionada a um peixe.
*   **Roncopatia:** Distúrbio do sono caracterizado por ruídos durante a respiração.
*   **Disfunção erétil:** Implícito pela menção de "dificultam o adormecer".

**Observações:**

*   A hiperuricemia é mencionada como uma possibilidade, mas não confirmada.
*   A "ferida no MS dto" é ambígua e pre

O modelo gerou uma resposta bem estruturada, apresentando as condições de forma clara, com pequenas explicações associadas. No geral, o conteúdo corresponde ao que está presente na nota, com uma leitura coerente e bem alinhada com o contexto clínico.

No entanto, reparei que o modelo também fez algumas interpretações mais subjetivas, assumindo como condição algo que não estava explicitamente descrito no texto (disfunção erétil). Isso reforça a importância de validar manualmente os resultados antes de aplicá-los em cenários reais.

O modelo demonstrou boa capacidade para interpretar e extrair informação clínica em português. Ainda assim, é importante ter atenção às inferências que podem não estar fundamentadas diretamente no texto original.


<div style="border-left: 6px solid #4CAF50; background-color: #f0fdf4; padding: 10px; border-radius: 5px;">
  <h4>📝 Exercise 3.3 - Named-Entity Recognition (NER) Model
  <p> These type of models are trained to identify entities in the clinical notes. Below we show how to use a NER model to extract entities, in this case conditions, from the clinical note.

  Considering the conditions identified, which are the main differences comparing to the output generated in the previous exercise?
  </p>
</div>

In [52]:
entity_extractor = pipeline('ner', model='portugueseNLP/medialbertina_pt-pt_900m_NER', aggregation_strategy="simple")
entities = entity_extractor(note_pt)

Device set to use cuda:0


In [53]:
diagnosis = [annotation['word'] for annotation in entities if annotation['entity_group'] == 'Diagnostico']
diagnosis

['DRC',
 'pielonefrite crónica litiásica (hiperuricémia?',
 'disfunção crónica do enxerto',
 'HTA secundária',
 'Adenocrcinoma da prostata',
 'Tumor Septissémia',
 'bactéria provocada por um peixe...',
 'Ferida no MS dto',
 'Roncopatia',
 'apneias']


- A maioria dos diagnósticos faz sentido e está de acordo com o texto.
- No entanto, alguns termos não são exatamente diagnósticos (ex: “bactéria provocada por um peixe” ou “ferida”), e outros como “apneias” foram identificados mesmo o texto dizendo que não havia apneias — ou seja, o modelo não entende negações.
- Comparando com o MedGemma (Ex. 3.2), o NER foi mais direto e extraiu só o que está escrito, sem tentar interpretar ou adivinhar.
- Por outro lado, o MedGemma foi capaz de perceber sintomas como insónia ou pernas inquietas, que o NER ignorou.

O modelo NER funciona muito bem para identificar termos clínicos diretamente, mas ainda precisa de ajuda humana para interpretar o contexto. Já o LLM como o MedGemma é melhor para entender o quadro geral, mas pode inventar ou confundir-se. Os dois se complementam bem.


**Model Description:**

MediAlbertina PT-PT 900M NER was created through fine-tuning of MediAlbertina PT-PT 900M on real European Portuguese EMRs that have been hand-annotated for the following entities:

- Diagnostico (D): All types of diseases and conditions following the ICD-10-CM guidelines.
- Sintoma (S): Any complaints or evidence from healthcare professionals indicating that a patient is experiencing a medical condition.
Medicamento (M): Something that is administrated to the patient (through any route), including drugs, specific food/drink, vitamins, or blood for transfusion.
- Dosagem (D): Dosage and frequency of medication administration.
ProcedimentoMedico (PM): Anything healthcare professionals do related to patients, including exams, moving patients, administering something, or even surgeries.
- SinalVital (SV): Quantifiable indicators in a patient that can be measured, always associated with a specific result. Examples include cholesterol levels, diuresis, weight, or glycaemia.
- Resultado (R): Results can be associated with Medical Procedures and Vital Signs. It can be a numerical value if something was measured (e.g., the value associated with blood pressure) or a descriptor to indicate the result (e.g., positive/negative, functional).
- Progresso (P): Describes the progress of patient’s condition. Typically, it includes verbs like improving, evolving, or regressing and mentions to patient’s stability.

https://huggingface.co/portugueseNLP/medialbertina_pt-pt_900m_NER